# Ejercicio 3 — Capital Económico con Cópulas
## Máster Executive en Finanzas Cuantitativas 2026 — AFI Global Education

**Objetivo:** Calcular el capital económico de un banco con exposición en Brasil, Chile, Francia, Italia y Alemania bajo tres enfoques: capital stand-alone, cópula Gaussiana y cópula t-Student.

**Nivel de confianza:** 95% | **Datos:** series_macro.xlsx (World Bank, 1991–2024)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3,'font.size':11})
SEED = 42
rng  = np.random.default_rng(SEED)

PAISES    = ['Brazil','Chile','France','Italy','Germany']
PAISES_ES = ['Brasil','Chile','Francia','Italia','Alemania']
MU     = np.array([10.0, 15.0, 18.0, 12.0, 19.0])
SIGMA  = np.array([ 0.75,  1.00,  3.00,  2.30,  4.00])
CONF   = 0.95
N_SIM  = 500_000
NU_T   = 5

os.makedirs('resultados', exist_ok=True)
print('Entorno OK | Seed =', SEED, '| N_SIM =', N_SIM)

## Paso 1 — Capital stand-alone

Con $L_i \sim \mathcal{N}(\mu_i, \sigma_i^2)$ independientes, el capital al 95% es:

$$CE_i = \mu_i + \sigma_i \cdot z_{0.95} = \mu_i + 1.6449\,\sigma_i$$

El capital total sin diversificación es la suma de los capitales individuales.

In [ ]:
z_alpha = stats.norm.ppf(CONF)
CE_standalone = MU + SIGMA * z_alpha
CE_standalone_total = CE_standalone.sum()

print(f'z_{{0.95}} = {z_alpha:.6f}')
print(f'\n{"País":<12} {"Media":>8} {"Std":>6} {"CE stand-alone":>16}')
print('-'*46)
for p, mu, s, ce in zip(PAISES_ES, MU, SIGMA, CE_standalone):
    print(f'{p:<12} {mu:>8.2f} {s:>6.2f} {ce:>16.4f}')
print('-'*46)
print(f'{"TOTAL":<12} {"":>8} {"":>6} {CE_standalone_total:>16.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
colores = ['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f']
bars = ax.bar(PAISES_ES, CE_standalone, color=colores, alpha=0.85, edgecolor='k')
for b, v in zip(bars, CE_standalone):
    ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontsize=10)
ax.set_ylabel('Capital Económico (u.m.)')
ax.set_title(f'Capital stand-alone por país (p95)  —  Total = {CE_standalone_total:.4f}')
fig.tight_layout()
fig.savefig('resultados/grafico_07_capital_standalone.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 2 — PCA por país: índice sintético macroeconómico

Para cada país se tienen 4 series anuales (PIB, inflación, desempleo, tipo de depósito) durante 1991–2024 (34 observaciones). El **primer componente principal** concentra la máxima varianza común y actúa como índice sintético del ciclo macroeconómico del país.

In [ ]:
# Ajusta la ruta si el notebook está en una subcarpeta
EXCEL_PATH = '../../../series_macro__1_.xlsx'

df_raw = pd.read_excel(EXCEL_PATH, sheet_name='Data')
year_cols = [c for c in df_raw.columns if str(c)[:4].isdigit()]
print(f'Series: {len(year_cols)} años ({year_cols[0][:4]}–{year_cols[-1][:4]})')
print(f'Shape: {df_raw.shape}')

In [ ]:
pca_indices = {}
pca_var_exp = {}

for pais_en in PAISES:
    df_pais = df_raw[df_raw['Country Name'] == pais_en][year_cols]
    assert df_pais.shape[0] == 4, f'{pais_en}: esperadas 4 series'
    X = df_pais.values.T.astype(float)          # (34, 4)
    X_std = StandardScaler().fit_transform(X)   # estandarizar
    pca = PCA(n_components=4, random_state=SEED)
    pca.fit(X_std)
    pca_indices[pais_en] = pca.transform(X_std)[:, 0]
    pca_var_exp[pais_en] = pca.explained_variance_ratio_[0]

print(f'{"País":<12} {"Var.explicada PC1":>20}')
print('-'*34)
for p_en, p_es in zip(PAISES, PAISES_ES):
    print(f'{p_es:<12} {pca_var_exp[p_en]:>18.1%}')

In [ ]:
years = [int(c[:4]) for c in year_cols]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (p_en, p_es) in enumerate(zip(PAISES, PAISES_ES)):
    ax = axes.flatten()[i]
    ax.plot(years, pca_indices[p_en], color=plt.cm.tab10(i), lw=2, marker='o', ms=3)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f'{p_es}  (PC1 = {pca_var_exp[p_en]:.1%})')
    ax.set_xlabel('Año'); ax.set_ylabel('PC1')
axes.flatten()[-1].set_visible(False)
fig.suptitle('Índice sintético macroeconómico — 1er componente principal (1991–2024)', fontsize=13)
fig.tight_layout()
fig.savefig('resultados/grafico_08_indices_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 3 — Matriz de correlaciones entre países

Se calcula la correlación de Pearson entre los índices sintéticos (PC1) de cada par de países. Esta correlación captura la co-movilidad macroeconómica histórica y será el input de la estructura de dependencia entre pérdidas.

In [ ]:
indices_matrix = np.column_stack([pca_indices[p] for p in PAISES])  # (34, 5)
corr_matrix    = np.corrcoef(indices_matrix.T)                       # (5, 5)

df_corr = pd.DataFrame(corr_matrix, index=PAISES_ES, columns=PAISES_ES)
print('Matriz de correlaciones (índices sintéticos PC1):')
print(df_corr.round(4).to_string())

eigenvalues = np.linalg.eigvalsh(corr_matrix)
assert np.all(eigenvalues >= -1e-8), 'Matriz NO semidefinida positiva'
print(f'\nValores propios: {eigenvalues.round(4)}')
print('Matriz valida (semidefinida positiva)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Correlacion de Pearson')
ax.set_xticks(range(5)); ax.set_xticklabels(PAISES_ES, rotation=30, ha='right')
ax.set_yticks(range(5)); ax.set_yticklabels(PAISES_ES)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{corr_matrix[i,j]:.3f}', ha='center', va='center', fontsize=10)
ax.set_title('Correlaciones entre indices sinteticos (PC1) — 1991–2024')
fig.tight_layout()
fig.savefig('resultados/grafico_09_heatmap_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 4 — Capital diversificado: Cópula Gaussiana

**Algoritmo:**
1. Simular $\mathbf{Z} \sim \mathcal{N}_5(\mathbf{0}, \Sigma)$ usando descomposición de Cholesky.
2. Como las marginales son normales: $L_i = \mu_i + \sigma_i Z_i$.
3. Pérdida total: $L_{\text{total}} = \sum_i L_i$.
4. Capital: percentil 95% de $L_{\text{total}}$.

In [ ]:
L_chol   = np.linalg.cholesky(corr_matrix)
Z_std    = rng.standard_normal((N_SIM, 5))
Z_corr   = Z_std @ L_chol.T
L_gauss  = MU + SIGMA * Z_corr
L_tot_g  = L_gauss.sum(axis=1)
CE_gauss = np.percentile(L_tot_g, 95)

print(f'Copula Gaussiana:')
print(f'  Media perdida total:  {L_tot_g.mean():.4f}  (teorica: {MU.sum():.4f})')
print(f'  Std  perdida total:   {L_tot_g.std():.4f}')
print(f'  Capital diversificado (p95): {CE_gauss:.4f}')

corr_emp = np.corrcoef(L_gauss.T)
print(f'  Max |Delta rho| = {np.abs(corr_emp - corr_matrix).max():.5f}')

## Paso 5 — Capital diversificado: Cópula t-Student

La cópula t captura **dependencia en las colas** (*tail dependence*): en escenarios extremos los países están más correlacionados, lo que produce un capital mayor que la cópula Gaussiana.

**Algoritmo:**
1. $\mathbf{Z} \sim \mathcal{N}_5(\mathbf{0}, \Sigma)$ correlacionada.
2. $\chi^2 \sim \chi^2_\nu$ independiente.
3. $\mathbf{T} = \mathbf{Z} / \sqrt{\chi^2/\nu}$ — t multivariante.
4. $U_i = F_{t_\nu}(T_i)$ — uniformes con estructura de dependencia t.
5. $L_i = \mu_i + \sigma_i \Phi^{-1}(U_i)$ — marginales normales.

In [ ]:
rng_t    = np.random.default_rng(SEED + 10)
Z_std_t  = rng_t.standard_normal((N_SIM, 5))
Z_corr_t = Z_std_t @ L_chol.T
chi2     = rng_t.chisquare(NU_T, size=N_SIM)
T_corr   = Z_corr_t / np.sqrt(chi2 / NU_T)[:, None]
U_t      = stats.t.cdf(T_corr, df=NU_T)
L_t      = MU + SIGMA * stats.norm.ppf(U_t)
L_tot_t  = L_t.sum(axis=1)
CE_t     = np.percentile(L_tot_t, 95)

print(f'Copula t-Student (nu={NU_T}):')
print(f'  Media perdida total:  {L_tot_t.mean():.4f}  (teorica: {MU.sum():.4f})')
print(f'  Std  perdida total:   {L_tot_t.std():.4f}')
print(f'  Capital diversificado (p95): {CE_t:.4f}')

## Paso 6 — Comparación y discusión

In [ ]:
ben_g = CE_standalone_total - CE_gauss
ben_t = CE_standalone_total - CE_t
pct_g = ben_g / CE_standalone_total * 100
pct_t = ben_t / CE_standalone_total * 100

print('=' * 60)
print(f'  Capital stand-alone total:          {CE_standalone_total:>10.4f}')
print(f'  Capital diversificado (Gauss):      {CE_gauss:>10.4f}')
print(f'  Capital diversificado (t, nu={NU_T}):   {CE_t:>10.4f}')
print('-' * 60)
print(f'  Beneficio div. (Gauss):             {ben_g:>10.4f}  ({pct_g:.1f}%)')
print(f'  Beneficio div. (t, nu={NU_T}):           {ben_t:>10.4f}  ({pct_t:.1f}%)')
print(f'  Extra capital copula t vs Gauss:    {CE_t - CE_gauss:>10.4f}')
print('=' * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: distribuciones de perdida total
ax = axes[0]
rango = (min(L_tot_g.min(), L_tot_t.min()), max(L_tot_g.max(), L_tot_t.max()))
ax.hist(L_tot_g, bins=150, density=True, alpha=0.5, color='steelblue',
        label=f'Gaussiana  (CE={CE_gauss:.2f})', range=rango)
ax.hist(L_tot_t, bins=150, density=True, alpha=0.5, color='orange',
        label=f't (nu={NU_T})  (CE={CE_t:.2f})', range=rango)
ax.axvline(CE_gauss, color='steelblue', lw=2.5, ls='--')
ax.axvline(CE_t,     color='darkorange', lw=2.5, ls='--')
ax.axvline(CE_standalone_total, color='red', lw=2, ls='-.',
           label=f'Stand-alone ({CE_standalone_total:.2f})')
ax.set_xlabel('Perdida total (u.m.)'); ax.set_ylabel('Densidad')
ax.set_title('Distribucion de perdidas agregadas'); ax.legend(fontsize=9)

# Panel 2: comparacion barras
ax2 = axes[1]
etiq = ['Stand-alone', 'Copula Gaussiana', f'Copula t (nu={NU_T})']
vals = [CE_standalone_total, CE_gauss, CE_t]
cols = ['#e15759','#4e79a7','#f28e2b']
bars2 = ax2.bar(etiq, vals, color=cols, alpha=0.85, edgecolor='k', width=0.4)
for b, v in zip(bars2, vals):
    ax2.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Capital Economico (u.m.)')
ax2.set_title('Comparacion de capitales (percentil 95%)')

fig.suptitle('Ejercicio 3 — Capital Economico Diversificado vs Stand-alone', fontsize=13)
fig.tight_layout()
fig.savefig('resultados/grafico_10_comparacion_capitales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada.')

## Discusion final

**Impacto de la diversificación:** El capital stand-alone asume correlacion perfecta entre paises. Al introducir la correlacion real (estimada via PCA), el capital diversificado es menor, reflejando que las perdidas extremas simultaneas en todos los paises son improbables.

**Diferencia entre copulas:** La copula t produce un capital mayor que la Gaussiana porque:
- Tiene colas mas pesadas → mayor probabilidad de perdidas extremas conjuntas.
- Captura *tail dependence*: en crisis, los paises tienden a correlacionarse mas.
- Es mas conservadora y relevante desde el punto de vista regulatorio (Basilea III/IV).

**Conclusion practica:** La eleccion del modelo de dependencia tiene un impacto significativo sobre el capital requerido. La copula Gaussiana subestima el riesgo en las colas respecto a la t-Student.